# Week 3 • Day 1: Google Colab, Hugging Face & Open-Source Generative AI (Lec 62–67)

Welcome to **Week 3 Day 1**! In this session, we transition from proprietary APIs to open-source foundation models hosted on Hugging Face.

### 🌟 Lectures Covered:
1. **Lec 62**: Google Colab 101, T4 Cloud GPUs & Runtime Management
2. **Lec 63**: Hugging Face Hub Authentication (`HF_TOKEN` with Write permissions)
3. **Lec 64**: Fast Single/Few-Step Latent Diffusion (`stabilityai/sdxl-turbo`)
4. **Lec 65**: Two-Stage High-Fidelity Diffusion (`sdxl-base-1.0` + `sdxl-refiner-1.0` 80/20 split)
5. **Lec 66**: Neural Text-to-Speech (`microsoft/speecht5_tts` with speaker embeddings)
6. **Lec 67**: Frontier Diffusion (`black-forest-labs/FLUX.1-schnell`) & GPU Cost Economics

---

## ☁️ Lec 62: Environment Setup & GPU Validation
Let's inspect our cloud GPU. On free Google Colab, you should be connected to an **NVIDIA Tesla T4**.

In [ ]:
# Core open-source generative AI stack
!pip install -q --upgrade transformers==4.56.2 diffusers==0.32.2 datasets==3.6.0 accelerate soundfile torch

In [ ]:
# Validate GPU connection
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('⚠️ Not connected to a GPU! Please change runtime type to T4 GPU (Runtime > Change runtime type).')
else:
    print(gpu_info)
    if 'Tesla T4' in gpu_info:
        print('✅ Success - Connected to NVIDIA Tesla T4 GPU!')
    elif 'A100' in gpu_info:
        print('🚀 Connected to high-performance NVIDIA A100 GPU!')
    else:
        print('Connected to GPU runtime.')

## 🤗 Lec 63: Hugging Face Hub Login
1. Go to https://huggingface.co/settings/tokens
2. Create a new token with **Write** permissions.
3. Add it to Colab Secrets (the Key icon on the left panel) under the name `HF_TOKEN`.
4. Run the cell below to log in.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token, add_to_git_credential=True)
    print('✅ Successfully authenticated with Hugging Face Hub!')
except Exception as e:
    print(f'Please set HF_TOKEN in Colab Secrets or log in interactively: {e}')
    login()

## ⚡ Lec 64: Fast Latent Diffusion with SDXL-Turbo
`stabilityai/sdxl-turbo` uses Adversarial Diffusion Distillation (ADD) to synthesize images in 1–4 steps without classifier-free guidance (`guidance_scale=0.0`).

In [ ]:
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe_turbo = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image_turbo = pipe_turbo(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image_turbo)

## 🎨 Lec 65: Two-Stage Diffusion (Base 80% + Refiner 20%)
Ensemble approach: The **Base model** establishes the global composition (first 80%), and the **Refiner model** polishes high-frequency details (last 20%).

In [ ]:
# Free up VRAM before loading the large base and refiner models
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from diffusers import DiffusionPipeline
import torch
from IPython.display import display

# 1. Load Base Model
base = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

# 2. Load Refiner Model (reuses text encoder 2 and VAE to save memory)
refiner = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    text_encoder_2=base.text_encoder_2,
    vae=base.vae,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
).to("cuda")

n_steps = 40
high_noise_frac = 0.8  # 80% base, 20% refiner
prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

# Stage 1: Generate coarse latent representation
latent_image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent"
).images

# Stage 2: Refine micro-details
final_image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=latent_image
).images[0]

display(final_image)

## 🎙️ Lec 66: Neural Text-to-Speech (SpeechT5 & Speaker Embeddings)
Uses `microsoft/speecht5_tts` paired with speaker embeddings from the `cmu-arctic-xvectors` dataset to synthesize customized vocal timbre.

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device="cuda")
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)

# Select a speaker vector (e.g. index 7306)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser(
    "Hi to an artificial intelligence engineer, on the way to mastery!",
    forward_params={"speaker_embeddings": speaker_embedding}
)

Audio(speech["audio"], rate=speech["sampling_rate"])

## 💎 Lec 67: Frontier FLUX.1-schnell & Cloud GPU Economics
*Note: FLUX.1-schnell requires an A100 GPU (Colab Pro) or substantial VRAM with `bfloat16`.*
Ensure you accepted model access at: https://huggingface.co/black-forest-labs/FLUX.1-schnell

In [ ]:
from datetime import datetime
import torch
from diffusers import FluxPipeline
from IPython.display import display

try:
    start = datetime.now()
    pipe_flux = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-schnell",
        torch_dtype=torch.bfloat16
    ).to("cuda")
    generator = torch.Generator(device="cuda").manual_seed(0)
    prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

    image = pipe_flux(
        prompt,
        guidance_scale=0.0,
        num_inference_steps=4,
        max_sequence_length=256,
        generator=generator
    ).images[0]
    display(image)
    stop = datetime.now()

    # GPU Cost Economics Calculation:
    seconds = (stop - start).total_seconds()
    units_per_hour = 5.37  # e.g. Colab A100 compute units per hour
    estimated_units = (units_per_hour / 3600) * seconds
    estimated_cost = estimated_units * (9.99 / 100)
    print(f"\n⏱️ Execution time: {seconds:.2f} seconds")
    print(f"💰 Estimated A100 cost for this generation: ${estimated_cost:.4f}")
    print("⚠️ Remember: You pay for all active session time, so shut down paid runtimes when idle!")
except Exception as e:
    print(f"FLUX requires A100 GPU or gated Hugging Face access approval: {e}")
    print("You can still calculate GPU economics using the formula above!")